# 🏥 Build a Clinical Knowledge RAG — Hands-on Guide

**Anaconda Desktop (Qwen3) · FAISS · FastAPI · secure-by-default (no pip, no HuggingFace Hub)**

> **Owner:** Federico Rubiano ([@federicorubiano](https://github.com/federicorubiano)) | **Status:** Active | **Estimated time:** 45–60 minutes

---

> ## ⚠️ MEDICAL DISCLAIMER
>
> **This notebook is for educational and demonstration purposes only.** Nothing it produces — generated text, citations, or clinical summaries — constitutes medical advice, diagnosis, or treatment. Always consult a qualified, licensed healthcare professional before any clinical decision. Do not use this tool in real patient care. The authors, contributors, and Anaconda, Inc. accept no liability for decisions made on the basis of this system's output.

---

## Audience

Python developers and data scientists who want to build a retrieval-augmented generation (RAG) system end to end on Anaconda's secure-by-default stack. No prior RAG experience needed.

## What you'll learn

By the end you will be able to:

1. **Build** a FAISS dense-vector index from a scraped corpus using Anaconda Desktop's local embedding server.
2. **Retrieve** the most relevant passages for a question with instruction-following embeddings.
3. **Generate** grounded, citation-enforced answers from a locally-served Qwen3-8B model.
4. **Serve** the pipeline through FastAPI and **evaluate** answer quality with a reproducible harness.

## Prerequisites

**Knowledge:** comfortable running Python; basic conda environments. RAG is taught here from scratch.

**Installed & running:** Anaconda Desktop, the project conda env (`environment-local.yml`), and the index built (`python scripts/build_index.py`). See the README for full setup.

**External dependencies:** Anaconda Desktop model server (*Tier 3, required*); Merck website for scraping (*Tier 2, sample corpus fallback in `data/raw/`*).

> ℹ️ **One server at a time.** Anaconda Desktop currently serves a single model. Run the **embedding** model for Sections 1–2, then switch to the **inference** model for Sections 3–5.

## Section 0 — Environment check

*Start state: you've created and activated the project conda env. Here we confirm the secure-by-default stack imports before building anything.*

In [ ]:
import sys, os

# Make `src/` importable when running from notebooks/
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")
print(f"Python: {sys.version.split()[0]}")

In [ ]:
# Verify the stack — every package is from the Anaconda `main` channel (no pip, no HuggingFace)
import importlib

packages = {
    'faiss': 'faiss-cpu',
    'fastapi': 'fastapi',
    'gradio': 'gradio',
    'evidently': 'evidently',
    'requests': 'requests',
    'numpy': 'numpy',
    'dotenv': 'python-dotenv',
    'tqdm': 'tqdm',
}

print('Package status:')
for module, pkg in packages.items():
    try:
        m = importlib.import_module(module)
        print(f"  ✅  {pkg:<16} {getattr(m, '__version__', 'installed')}")
    except ImportError:
        print(f"  ❌  {pkg:<16} NOT INSTALLED")

**✅ Checkpoint 0:** every package shows a green check. There is **no** `vllm`, `sentence-transformers`, or `rank-bm25` — inference and embeddings are served locally by Anaconda Desktop, and retrieval is dense-only.

Expected output (versions will vary):
```text
Package status:
  ✅  faiss-cpu        1.8.0
  ✅  fastapi          0.115.0
  ✅  gradio           4.44.0
  ✅  evidently        0.4.x
  ✅  requests         2.32.0
  ✅  numpy            1.26.x
  ✅  python-dotenv    1.0.x
  ✅  tqdm             4.66.x
```

## Section 1 — How the pipeline works

The RAG flow you're about to build:

```
User question
    │
    ▼
DenseRetriever
    └─ Qwen3-Embedding-4B (query instruction prefix)
         → FAISS dense search → top-k chunks
         [no cross-encoder — instruction-following asymmetry handles ranking]
    │
    ▼
DesktopClient (Qwen3-8B via Anaconda Desktop)
    │   System prompt: citation rules + structure enforcement
    │   User prompt:   retrieved context + question
    ▼
Answer + Citations + Medical Disclaimer
    └─ served by FastAPI /query
```

**Why this design?** Dense retrieval finds passages by *meaning*, not keywords. Qwen3-Embedding's instruction-following design ranks query/document relevance natively, so we drop the separate reranker the V1 system used — fewer moving parts, fully local.

## Section 2 — Build the index (embeddings)

*Start state: in Anaconda Desktop, start the **embedding** model server (`Qwen3-Embedding-4B`) at `localhost:8080`.*

We scrape the Merck Manual (if needed) and embed every chunk into a FAISS index. If `data/index/merck.faiss` already exists, this cell just confirms it.

In [ ]:
import subprocess, os

index_path = os.path.join(project_root, 'data', 'index', 'merck.faiss')
raw_dir = os.path.join(project_root, 'data', 'raw')

if not os.path.exists(index_path):
    print('No index found — scraping + building (this calls the Desktop embedding server)...')
    # Tip: you can also run these in a terminal:
    #   python scripts/scraper.py
    #   python scripts/build_index.py
    subprocess.run([sys.executable, os.path.join(project_root, 'scripts', 'scraper.py')], check=False)
    subprocess.run([sys.executable, os.path.join(project_root, 'scripts', 'build_index.py')], check=False)
else:
    print('Index already present — skipping rebuild.')

print('\nraw topics:', len([f for f in os.listdir(raw_dir) if f.endswith('.txt')]) if os.path.isdir(raw_dir) else 0)
print('index exists:', os.path.exists(index_path))

**✅ Checkpoint 2:** `data/index/merck.faiss` and `data/index/chunks.json` exist and a chunk count is reported.

Expected output (numbers will vary):
```text
Index already present — skipping rebuild.

raw topics: 7
index exists: True
```

> ℹ️ If `build_index.py` errors with **exit code 133**, that's the documented Anaconda Desktop embedding crash on chunks >512 tokens. Keep `CHUNK_SIZE_WORDS=200` in `.env` (see README → Known issues).

## Section 3 — Retrieve relevant passages

*Start state: the index exists (Section 2) and the embedding server is running. We load the retriever and run a query.*

In [ ]:
from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))

from src.retriever import DenseRetriever  # (HybridRetriever is a backwards-compatible alias)

_desktop = os.getenv('DESKTOP_API_URL', 'http://localhost:8080/v1')
retriever = DenseRetriever(
    faiss_path=os.path.join(project_root, 'data/index/merck.faiss'),
    chunks_path=os.path.join(project_root, 'data/index/chunks.json'),
    api_url=os.getenv('EMBEDDING_API_URL', _desktop),
    embedding_model=os.getenv('EMBEDDING_MODEL', 'Qwen3-Embedding-4B'),
    top_k=int(os.getenv('TOP_K', 5)),
)
print(f'Retriever loaded: {len(retriever.chunks)} chunks indexed')

In [ ]:
DEMO_QUERY = 'What is the protocol for managing sepsis in a critical care unit?'
print(f'Query: {DEMO_QUERY}\n--- Retrieving...\n')

chunks = retriever.retrieve(DEMO_QUERY)

for i, c in enumerate(chunks, 1):
    print(f'[{i}] {c.section}  (score={c.score:.4f})')
    print(f'    merckmanuals.com{c.url}')
    print(f'    {c.text[:150].strip()}...\n')

**✅ Checkpoint 3:** you get `top_k` chunks, each with a section name, a similarity score, a source URL, and a text preview — and the top results are clearly about sepsis.

Expected output (illustrative):
```text
[1] Sepsis and Septic Shock  (score=0.83)
    merckmanuals.com/professional/critical-care-medicine/sepsis-and-septic-shock
    Sepsis is a clinical syndrome of life-threatening organ dysfunction...
```

## Section 4 — Generate a grounded answer

*Start state: switch Anaconda Desktop to the **inference** model server (`Qwen3-8B`). (Desktop serves one model at a time.) We pass the retrieved chunks to the model with citation rules enforced.*

In [ ]:
from src.vllm_client import VLLMClient  # backwards-compatible alias for DesktopClient

llm = VLLMClient(
    base_url=os.getenv('INFERENCE_API_URL', _desktop),
    model=os.getenv('INFERENCE_MODEL', 'Qwen3-8B'),
)
print(f'Inference endpoint : {llm.base_url}')
print(f'Model              : {llm.model}')

In [ ]:
result = llm.generate(question=DEMO_QUERY, chunks=chunks, max_tokens=512, temperature=0.1)

print('=' * 70)
print('ANSWER')
print('=' * 70)
print(result['answer'])
print(f"\nTokens used: {result.get('usage')}")

**✅ Checkpoint 4:** the answer is grounded in the retrieved passages, ends with a **CITATIONS** section listing the Merck sources used, and includes the medical disclaimer. If the context didn't cover the question, the model says so rather than inventing an answer.

Expected shape (illustrative):
```text
Sepsis management in critical care follows...
1. Early recognition and source control...

CITATIONS
[Sepsis and Septic Shock] — merckmanuals.com/professional/...

⚠️ Medical disclaimer: ...
```

## Section 5 — Serve and evaluate

*Start state: in a separate terminal, start the API — `uvicorn src.api:app --port 8000` — with the inference server running. Then run the cells below.*

In [ ]:
import requests

API_URL = os.getenv('API_URL', 'http://localhost:8000')
health = requests.get(f'{API_URL}/health').json()
print('Health check:')
for k, v in health.items():
    print(f'  {k:<18} {v}')

In [ ]:
BENCHMARK_QUERIES = [
    'What is the protocol for managing sepsis in a critical care unit?',
    'What are the common symptoms for appendicitis, and can it be cured via medicine?',
    'What are the effective treatments for sudden patchy hair loss on the scalp?',
    'What treatments are recommended for traumatic brain injury?',
    'What are the precautions and treatment steps for a leg fracture during a hiking trip?',
]

import time
for i, q in enumerate(BENCHMARK_QUERIES, 1):
    t0 = time.perf_counter()
    resp = requests.post(f'{API_URL}/query', json={'question': q, 'max_tokens': 512}).json()
    elapsed = int((time.perf_counter() - t0) * 1000)
    sources = [s['section'] for s in resp.get('sources', [])]
    print(f"[{i}] {q[:55]}...")
    print(f"     latency: {resp.get('latency_ms', elapsed)} ms | sources: {sources}")
    print(f"     answer : {resp.get('answer','')[:110].strip()}...\n")

In [ ]:
# Reproducible scoring — heuristic, no LLM-as-judge
import subprocess, json
import pandas as pd

subprocess.run(
    [sys.executable, os.path.join(project_root, 'eval', 'run_eval.py'),
     '--api-url', API_URL,
     '--output', os.path.join(project_root, 'eval', 'results.json'),
     '--report'],
    check=False,
)

results_path = os.path.join(project_root, 'eval', 'results.json')
if os.path.exists(results_path):
    data = json.load(open(results_path))
    print('Aggregate scores:')
    agg = {k: v for k, v in data['aggregate'].items() if k not in ('queries_scored', 'queries_failed', 'avg_latency_ms')}
    display(pd.DataFrame([agg]).T.rename(columns={0: 'score'}))
else:
    print('No results yet — make sure the API is running, then re-run this cell.')

**✅ Checkpoint 5:** `/health` returns the model name and `index_loaded: true`; all 5 benchmark queries return grounded answers with sources; and `eval/report.html` is generated with scores for groundedness, relevance, citation rate, disclaimer presence, and overall.

If any query returns an error, check that **both** the inference server (Desktop) and the API (`uvicorn`) are running.

## Extension challenges (optional)

Make it your own — each is optional and open-ended:

- **Add a topic.** Add a Merck URL to `scripts/scraper.py`, re-run scrape + `build_index.py`, and ask about it. Did retrieval surface the new content?
- **Tune `TOP_K`.** Try 3 then 8 in `.env`, re-run the eval, and compare groundedness vs. latency.
- **Swap the model.** Point `INFERENCE_MODEL` at another chat model in your Desktop catalog and compare answers on the 5 benchmarks — no code changes.
- **Bring your own corpus.** Replace the scraper with a different public dataset; the rest of the pipeline is domain-agnostic.

## Recap

You built a clinical RAG end to end — index → retrieve → generate → serve → evaluate — entirely on Anaconda's secure-by-default stack: every package from the `main` channel, every model weight from Anaconda Desktop's curated catalog, zero pip and zero HuggingFace Hub calls at runtime.

**Where to go next:** [Anaconda AI Navigator](https://www.anaconda.com/docs/tools/ai-navigator/main) · [FAISS wiki](https://github.com/facebookresearch/faiss/wiki) · [Evidently docs](https://docs.evidentlyai.com) · README → *Build it yourself* for the full setup and Known issues.

In [ ]:
print('Guide complete ✅')
print('\nStack used — all from Anaconda main, served locally:')
for pkg, desc in [
    ('Anaconda Desktop', 'Local Qwen3-8B (inference) + Qwen3-Embedding-4B (embeddings)'),
    ('FAISS',            'Dense vector similarity search'),
    ('FastAPI',          'Production REST API'),
    ('Evidently AI',     'Reproducible RAG evaluation'),
    ('Gradio',           'Interactive demo UI'),
    ('requests',         'All model API calls — no SDKs'),
]:
    print(f'  • {pkg:<18} {desc}')